In [ ]:
print("Startup Job Finder – discovery layer ready")

Setup

In [ ]:
from tavily import TavilyClient
from dotenv import load_dotenv
import os

load_dotenv()
client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

Company discovery query

In [ ]:
query = (
  '"AI startup" '
  '("about us" OR "company" OR "who we are") '
  '-news -blog -article -wikipedia -linkedin -medium'
)

response = client.search(
    query=query,
    search_depth="advanced",
    max_results=20
)

for r in response["results"]:
    print("TITLE:", r["title"])
    print("URL:", r["url"])
    print("----")


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

CAREER_KEYWORDS = [
    "career",
    "job",
    "join",
    "work with",
    "hiring",
]

BAD_HINTS = ["contact", "about", "privacy", "terms", "press", "blog"]

def looks_like_careers(text: str, href: str) -> bool:
    t = text.lower()
    h = href.lower()

    # reject anchors like "#Contact"
    if h.startswith("#"):
        return False

    # reject obvious non-careers pages
    if any(b in t or b in h for b in BAD_HINTS):
        return False

    # accept careers-ish
    return any(k in t or k in h for k in CAREER_KEYWORDS)


def find_careers_url(homepage: str):
    try:
        resp = requests.get(homepage, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")

        for a in soup.find_all("a", href=True):
            text = (a.get_text() or "").lower()
            href = a["href"].lower()

            if looks_like_careers(text, href):
                url = urljoin(homepage, href)
                return url.rstrip("/")

    except Exception as e:
        print("ERROR:", homepage, e)

    return None


In [ ]:
companies = [
    "https://alphaai.biz",
    "https://ai-nation.de",
    "https://apera.ai",
]

for c in companies:
    print(c, "→", find_careers_url(c))


In [15]:
import requests
from bs4 import BeautifulSoup

def extract_careers_text(careers_url: str) -> str:
    resp = requests.get(careers_url, timeout=10)
    soup = BeautifulSoup(resp.text, "html.parser")

    # remove junk
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")

    # basic cleanup
    lines = [line.strip() for line in text.splitlines()]
    lines = [l for l in lines if len(l) > 30]

    return "\n".join(lines[:200])  # cap: first ~200 lines


In [16]:
careers_pages = [
    ("Alpha AI", "https://alphaai.biz/careers"),
    ("Apera AI", "https://apera.ai/careers"),
]

for name, url in careers_pages:
    print("====", name, "====")
    text = extract_careers_text(url)
    print(text[:1000])
    print("\n")


==== Alpha AI ====
Alpha AI - Opportunities, Roles
Among India’s top 10 Companies in the Field of Applied Artificial Intelligence
Want to join our AI Research and Development team?
BENEFITS OF THE INTERNSHIPS / OPPORTUNITY
BENEFITS OF THE INTERNSHIPS / OPPORTUNITY
At the moment we are only accepting applications for
React Native / Flutter App Developer
Full Stack Web Developer - Intern
: Build web apps with MEAN/MERN stack, APIs (Flask/Django), PostgreSQL (pgvector), and offline LLMs via WebLLM.
: JavaScript, HTML, CSS, Python, PostgreSQL, Docker.
: Pursuing/recent CS degree, strong problem-solving skills.
React Native / Flutter App Developer - Intern
: Develop mobile apps with Flutter/React Native, offline AI models, and PostgreSQL/SQLite.
: Flutter/React Native, Python, PostgreSQL/SQLite, TensorFlow Lite, Docker.
: Pursuing/recent CS degree, passion for mobile development.
While we are a bootstrapped company and can offer a
, we value the opportunity for interns to gain hands-on expe

In [17]:
JOB_HINTS = [
    "engineer", "developer", "scientist", "researcher", "intern",
    "backend", "full stack", "full-stack", "machine learning", "ml", "ai"
]

def extract_jobish_lines(text: str, window: int = 4) -> str:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    keep = set()

    for i, line in enumerate(lines):
        low = line.lower()
        if any(h in low for h in JOB_HINTS):
            for j in range(max(0, i - window), min(len(lines), i + window + 1)):
                keep.add(j)

    out = [lines[i] for i in sorted(keep)]
    return "\n".join(out)
